# RFC Simulation Lab: Triad-First Deterministic Closure

This notebook is the current simulator for **Recursive Fractal Cosmology: A Generative Ontology of Existence**.

It keeps the older downstream module family **A-F and H-Q**, but updates the simulator around the current canonical pipeline:

```text
CIF-QV-RFL triad
-> Module G: deterministic triadic closure
-> Module R: triad-grouped global closure audit
-> Module N V2: dimensional projection bridge
-> downstream physical-projection screen
-> dimensionless validation
-> Module S: one-anchor SI bridge
-> Module T: dimensionless coupling map
```

Core rule:

```text
Modules consume the frozen packet.
Modules do not choose or retune the frozen packet.
```

Deprecated modules such as `G_legacy`, `N_legacy`, and `R_legacy` are preserved only for historical comparison.

In [ ]:
# Core imports
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, HTML
from ipywidgets import Dropdown, Button, VBox, Output, Layout

# Optional json5 support for older config files.
try:
    import json5
except Exception:
    json5 = None

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

## 1. Load configuration files

The notebook looks for both:

```text
SimulationConfigs.json
Module_G_R_N_S_T_FrozenPacket.json
```

in the current folder, the repo root, or `notebooks/`.

In [ ]:
def load_json_file(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        if json5 is None:
            raise
        return json5.loads(text)


def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


CONFIG_CANDIDATES = [
    Path("SimulationConfigs.json"),
    Path("notebooks") / "SimulationConfigs.json",
    Path.cwd() / "SimulationConfigs.json",
    Path.cwd() / "notebooks" / "SimulationConfigs.json"
]

PACKET_CANDIDATES = [
    Path("Module_G_R_N_S_T_FrozenPacket.json"),
    Path("notebooks") / "Module_G_R_N_S_T_FrozenPacket.json",
    Path.cwd() / "Module_G_R_N_S_T_FrozenPacket.json",
    Path.cwd() / "notebooks" / "Module_G_R_N_S_T_FrozenPacket.json"
]

config_path = find_first_existing(CONFIG_CANDIDATES)
packet_path = find_first_existing(PACKET_CANDIDATES)

if config_path is None:
    raise FileNotFoundError("Could not find SimulationConfigs.json in repo root or notebooks/.")

all_configs = load_json_file(config_path)
if not isinstance(all_configs, list):
    raise ValueError("SimulationConfigs.json should be a top-level JSON array of module configs.")

frozen_packet = None
if packet_path is not None:
    frozen_packet = load_json_file(packet_path)

print(f"Loaded module configs from: {config_path}")
if packet_path is not None:
    print(f"Loaded frozen packet from: {packet_path}")
else:
    print("No frozen-packet JSON found. Using values embedded in SimulationConfigs.json.")

print(f"Number of module configs loaded: {len(all_configs)}")

## 2. Core utilities

In [ ]:
def module_key(cfg):
    return str(cfg.get("module", cfg.get("id", "UNKNOWN")))


def get_config(selected_module):
    for cfg in all_configs:
        if module_key(cfg) == selected_module:
            return cfg
    raise KeyError(f"Module {selected_module} not found in SimulationConfigs.json")


def get_active_g_packet():
    # Prefer frozen-packet file.
    if isinstance(frozen_packet, dict):
        g = frozen_packet.get("moduleG_deterministicTriadicClosure", {})
        if isinstance(g, dict) and "packet" in g:
            return g["packet"]

    # Fall back to active Module G inside SimulationConfigs.json.
    for cfg in all_configs:
        if cfg.get("module") == "G" and "packet" in cfg:
            return cfg["packet"]

    # Safe fallback.
    return {
        "delta": 4.6692,
        "cycleLength": 60,
        "alpha": 0.0256831,
        "phaseDepthK": 2,
        "nu": 0.00420784,
        "epsilon": 0.000108071,
        "lambdaNormalized": 0.489442,
        "nClosure": 18,
        "nFullCanonical": 40
    }


G_PACKET = get_active_g_packet()


def packet_value(name, default=None):
    return G_PACKET.get(name, default)


def get_delta_alpha_nu_n(cfg=None):
    cfg = cfg or {}
    delta = cfg.get("delta", packet_value("delta", 4.6692))
    alpha = cfg.get("alpha", packet_value("alpha", 0.0256831))
    nu = cfg.get("nu", packet_value("nu", 0.00420784))
    n = cfg.get("nFullCanonical", packet_value("nFullCanonical", 40))
    return float(delta), float(alpha), float(nu), int(n)


def time_grid(cfg, default_end=None):
    dt = float(cfg.get("dt", 0.1))
    t_range = cfg.get("t_range", [0, default_end or packet_value("cycleLength", 60)])
    tmin, tmax = float(t_range[0]), float(t_range[1])
    if tmax <= tmin:
        tmax = tmin + (default_end or 60)
    vals = np.arange(tmin, tmax, dt)
    if len(vals) < 2:
        vals = np.linspace(tmin, tmax, 100)
    return vals, dt, tmin, tmax


def recursive_kernel(t_vals, delta=None, alpha=None, nu=None, n=None, mode="cos"):
    delta = packet_value("delta", 4.6692) if delta is None else delta
    alpha = packet_value("alpha", 0.0256831) if alpha is None else alpha
    nu = packet_value("nu", 0.00420784) if nu is None else nu
    n = packet_value("nFullCanonical", 40) if n is None else n

    y = np.zeros_like(t_vals, dtype=float)
    for j in range(1, int(n) + 1):
        weight = (delta ** (-j)) * np.exp(-alpha * j * t_vals)
        if mode == "sin":
            y += weight * np.sin(j * t_vals + nu)
        elif mode == "rebirth":
            y += weight * np.sin(j * t_vals + j ** 2)
        else:
            y += weight * np.cos(j * t_vals + nu)
    return y


def recursive_entropy(t_vals, delta=None, alpha=None, n=None):
    delta = packet_value("delta", 4.6692) if delta is None else delta
    alpha = packet_value("alpha", 0.0256831) if alpha is None else alpha
    n = packet_value("nClosure", 18) if n is None else n

    s = np.zeros_like(t_vals, dtype=float)
    for j in range(1, int(n) + 1):
        s += j * np.log(delta) * (delta ** (-j)) * np.exp(-alpha * j * t_vals)
    return s


def safe_json(data):
    return json.dumps(data, indent=2, ensure_ascii=False)


def display_dict(title, data):
    display(Markdown(f"### {title}"))
    display(HTML(f"<pre style='white-space: pre-wrap; font-size: 12px'>{safe_json(data)}</pre>"))


def bar_chart(labels, values, title, ylabel="Value", log=False):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values)
    if log:
        plt.yscale("log")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


def print_claim_boundary(cfg):
    status = cfg.get("currentStatus", "unspecified")
    role = cfg.get("canonicalRole", "unspecified")
    may_retune = cfg.get("mayRetuneFrozenPacket", None)
    display(Markdown(f"**Current status:** `{status}`"))
    display(Markdown(f"**Canonical role:** {role}"))
    if may_retune is not None:
        display(Markdown(f"**May retune frozen packet:** `{may_retune}`"))
    if "claimBoundary" in cfg:
        display(Markdown(f"**Claim boundary:** {cfg['claimBoundary']}"))


display_dict("Active frozen Module G packet", G_PACKET)

## 3. Active canonical module runners: G, R, N V2, S, T

In [ ]:
def run_module_G(cfg):
    display(Markdown("## Module G: Deterministic Triadic Closure"))
    print_claim_boundary(cfg)

    packet = cfg.get("packet", G_PACKET)
    display_dict("Frozen deterministic packet", packet)

    delta = packet["delta"]
    cycle = packet["cycleLength"]
    alpha_expected = math.log(delta) / cycle
    nu_expected = packet["phaseDepthK"] * delta ** (-4)
    epsilon_expected = packet["alpha"] * packet["nu"]

    checks = {
        "empiricalTargetsUsed": cfg.get("empiricalTargetsUsed", False),
        "parameterSearchPerformed": cfg.get("parameterSearchPerformed", False),
        "mcmcUsed": cfg.get("mcmcUsed", False),
        "nutsUsed": cfg.get("nutsUsed", False),
        "alpha_expected_Log_delta_over_cycle": alpha_expected,
        "alpha_packet": packet["alpha"],
        "alpha_abs_error": abs(alpha_expected - packet["alpha"]),
        "nu_expected_phaseDepthK_delta_minus4": nu_expected,
        "nu_packet": packet["nu"],
        "nu_abs_error": abs(nu_expected - packet["nu"]),
        "epsilon_expected_alpha_times_nu": epsilon_expected,
        "epsilon_packet": packet["epsilon"],
        "epsilon_abs_error": abs(epsilon_expected - packet["epsilon"])
    }
    display_dict("Deterministic closure checks", checks)

    breakdown = cfg.get("closureBreakdownAtN18", {})
    if breakdown:
        labels = list(breakdown.keys())
        values = [breakdown[k] for k in labels]
        bar_chart(labels, values, "Module G closure breakdown at n = 18")

    t_vals = np.linspace(0, packet["cycleLength"], 800)
    psi = recursive_kernel(t_vals, packet["delta"], packet["alpha"], packet["nu"], packet["nFullCanonical"])
    S = recursive_entropy(t_vals, packet["delta"], packet["alpha"], packet["nClosure"])

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi, label="psi(t; nu)")
    plt.plot(t_vals, S, label="S_rec(t)")
    plt.title("Frozen Module G kernel and recursive entropy")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    symbolic = cfg.get("symbolicOutputs", {})
    if symbolic:
        display_dict("Symbolic outputs", symbolic)


def run_module_R(cfg):
    display(Markdown("## Module R: Triad-Grouped Global Closure Audit"))
    print_claim_boundary(cfg)

    core = {
        "rawRFLResidualScore": cfg.get("rawRFLResidualScore"),
        "sourceCoupledRFLResidualScore": cfg.get("sourceCoupledRFLResidualScore"),
        "residualImprovement": cfg.get("residualImprovement"),
        "cpResidual": cfg.get("cpResidual"),
        "tailN18": cfg.get("tailN18"),
        "tailN40": cfg.get("tailN40"),
        "bestLagV1": cfg.get("bestLagV1"),
        "bestLagCorrelationV1": cfg.get("bestLagCorrelationV1")
    }
    display_dict("Module R core audit values", core)

    v2 = cfg.get("standardizedResidualAuditV2", {})
    if v2:
        display_dict("Standardized residual audit V2", v2)
        labels = ["raw standardized", "source-coupled V2"]
        values = [v2.get("rawStandardizedResidualScore", 0), v2.get("sourceCoupledRFLResidualScoreV2", 0)]
        bar_chart(labels, values, "Module R residual improvement")

    triad = cfg.get("interpretiveTriadFractions_WithDarkKernelBridge", {})
    if triad:
        labels = list(triad.keys())
        values = [triad[k] for k in labels]
        bar_chart(labels, values, "Module R interpretive triad fractions with dark-kernel bridge")

    source = cfg.get("sourcePowerFractionsV2", {})
    if source:
        labels = list(source.keys())
        values = [source[k] for k in labels]
        bar_chart(labels, values, "Module R source power fractions V2")


def run_module_N(cfg):
    display(Markdown("## Module N V2: Dimensional Projection Bridge"))
    print_claim_boundary(cfg)

    display_dict("Projection factors", cfg.get("projectionFactors", {}))
    display_dict("Symbolic constants stable across n", cfg.get("symbolicConstantsStableAcrossN", {}))
    display_dict("Projected values", cfg.get("canonicalProjectedValuesStableAcrossN", {}))
    display_dict("Identity residuals", cfg.get("identityResiduals", {}))

    proj = cfg.get("canonicalProjectedValuesStableAcrossN", {})
    if proj:
        m = proj.get("symbolicMassProjected")
        E = proj.get("meanEnergyProjected")
        alpha_em = proj.get("alphaEMProjected_inverseEnergy")
        alpha_g = proj.get("alphaGProjected_massSquared")
        Lambda = proj.get("LambdaProjected_inverseEnergyCycle")
        T = proj.get("cycleTimeProjected")
        checks = {}
        if m is not None and alpha_g is not None:
            checks["alphaG_over_mSquared"] = alpha_g / (m ** 2)
        if E is not None and alpha_em is not None:
            checks["alphaEM_times_E"] = alpha_em * E
        if E is not None and Lambda is not None and T is not None:
            checks["Lambda_times_E_times_T"] = Lambda * E * T
        display_dict("Identity checks recomputed in notebook", checks)

        labels = ["m projected", "E projected", "alpha_G projected", "Lambda projected"]
        values = [
            proj.get("symbolicMassProjected", 0),
            proj.get("meanEnergyProjected", 0),
            proj.get("alphaGProjected_massSquared", 0),
            proj.get("LambdaProjected_inverseEnergyCycle", 0)
        ]
        bar_chart(labels, values, "Module N V2 projected internal quantities")


def run_module_S(cfg):
    display(Markdown("## Module S: One-Anchor SI / Laboratory Bridge"))
    print_claim_boundary(cfg)
    display_dict("Unit bridge", cfg.get("unitBridge", {}))
    display_dict("Anchor check", cfg.get("anchorCheck", {}))

    bridge = cfg.get("unitBridge", {})
    labels = []
    values = []
    for k in ["RFC_energy_unit_MeV", "RFC_energy_unit_J", "RFC_time_unit_s", "RFC_length_unit_m", "projectedCycleTimeSI_s", "projectedCycleLengthSI_m"]:
        if k in bridge:
            labels.append(k)
            values.append(bridge[k])
    if labels:
        bar_chart(labels, values, "Module S SI bridge scales, log axis", log=True)


def run_module_T(cfg):
    display(Markdown("## Module T: Dimensionless Coupling Map"))
    print_claim_boundary(cfg)
    display_dict("Canonical map", cfg.get("canonicalMap", {}))
    display_dict("Output", cfg.get("output", {}))

    cmap = cfg.get("canonicalMap", {})
    out = cfg.get("output", {})
    raw = cmap.get("alpha_N_raw_inverse", cfg.get("rawInverseEnergyCoupling", {}).get("alpha_N_raw_inverse"))
    factor = cmap.get("screenFactor_F_T")
    mapped = out.get("alpha_T_inverse")
    ref = out.get("reference_alpha_inverse")

    if raw is not None and factor is not None:
        recomputed = raw * factor
        display_dict("Notebook recomputation", {
            "raw_inverse_energy_coupling": raw,
            "screen_factor": factor,
            "recomputed_alpha_T_inverse": recomputed,
            "stored_alpha_T_inverse": mapped,
            "reference_alpha_inverse_after_the_fact": ref
        })

    labels = ["raw inverse", "mapped inverse", "reference inverse"]
    values = [raw or 0, mapped or 0, ref or 0]
    bar_chart(labels, values, "Module T raw vs mapped vs reference inverse coupling")

## 4. Deprecated legacy module runners

In [ ]:
def run_module_G_legacy(cfg):
    display(Markdown("## G_legacy: Deprecated MCMC/NUTS Parameter Assimilation"))
    print_claim_boundary(cfg)
    display(Markdown("**Warning:** This module is historical only. It must not be used to choose the current RFC packet."))

    prior = cfg.get("prior_ranges", {})
    if "epsilon" in prior:
        eps_min, eps_max = prior["epsilon"]
    else:
        eps_min, eps_max = 1e-5, 5e-4

    eps_vals = np.linspace(eps_min, eps_max, 200)
    Yp = 0.246 + 0.015 * np.exp(-1000 * eps_vals)
    Df = 2.4 + 0.05 * np.sin(500 * eps_vals)
    Echo = 18 + 2 * np.cos(200 * eps_vals)

    plt.figure(figsize=(9, 4))
    plt.plot(eps_vals, Yp, label="Y_p(epsilon)")
    plt.plot(eps_vals, Df, label="D_f(epsilon)")
    plt.plot(eps_vals, Echo, label="EchoDelay(epsilon)")
    plt.title("Deprecated legacy Module G: historical MCMC observable curves")
    plt.xlabel("epsilon")
    plt.legend()
    plt.tight_layout()
    plt.show()

    display_dict("Legacy best-fit values", cfg.get("best_fit", {}))
    display_dict("Deprecation reason", {
        "reasonForDeprecation": cfg.get("reasonForDeprecation"),
        "oldFlow": cfg.get("oldFlow"),
        "currentFlow": cfg.get("currentFlow")
    })


def run_module_N_legacy(cfg):
    display(Markdown("## N_legacy: Deprecated Symbolic Constants Module"))
    print_claim_boundary(cfg)
    display(Markdown("**Warning:** Module N V2 replaces this historical module for the current RFC framework."))

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    delta = cfg.get("delta", 4.669)
    alpha = cfg.get("alpha", 0.01)
    nu_values = cfg.get("nu_values", [0.006, 0.007, 0.008])

    rows = []
    for nu in nu_values:
        psi = recursive_kernel(t_vals, delta, alpha, nu, 40)
        dpsi = np.gradient(psi, dt)
        E = dpsi ** 2 + psi ** 2
        m = np.sqrt(np.mean(psi ** 2))
        rows.append({
            "nu": nu,
            "alpha_G": m ** 2,
            "alpha_EM_raw": 1 / np.mean(E),
            "Lambda_symbolic_raw": 1 / np.sum(E)
        })
    display_dict("Legacy Module N recomputed symbolic values", rows)


def run_module_R_legacy(cfg):
    display(Markdown("## R_legacy: Deprecated Fractal Integral Cosmology Module"))
    print_claim_boundary(cfg)
    display(Markdown("**Warning:** Active Module R is now the triad-grouped global closure audit."))

    t_vals, dt, _, _ = time_grid(cfg, default_end=120)
    delta = cfg.get("delta", 4.669)
    alpha = cfg.get("alpha", 0.01)
    nu_values = cfg.get("nu_values", [0.003, 0.006, 0.009])

    plt.figure(figsize=(9, 4))
    for nu in nu_values:
        psi = recursive_kernel(t_vals, delta, alpha, nu, 40)
        plt.plot(t_vals, psi, label=f"nu={nu}")
    plt.title("Legacy Module R fractal modal fields")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    display_dict("Legacy R fields", cfg.get("fields", {}))


def run_generic_legacy_warning(cfg):
    display(Markdown(f"## {cfg.get('module')}: {cfg.get('description', '')}"))
    print_claim_boundary(cfg)
    display(Markdown("**Warning:** Deprecated or historical module. Preserved for development history only."))
    display_dict("Legacy configuration", cfg)

## 5. Downstream module runners A-F and H-Q

In [ ]:
def run_module_A(cfg):
    display(Markdown("## Module A: Downstream cosmology and dark-sector projection"))
    print_claim_boundary(cfg)

    t_vals, dt, tmin, tmax = time_grid(cfg, default_end=60)
    delta, alpha, _, _ = get_delta_alpha_nu_n(cfg)
    n_closure = int(cfg.get("nClosure", packet_value("nClosure", 18)))
    lam = cfg.get("lambdaNormalized", packet_value("lambdaNormalized", 0.489442))

    a = np.zeros_like(t_vals)
    adot = np.zeros_like(t_vals)
    a[0] = cfg.get("initial_conditions", {}).get("a(0)", 1.0)
    adot[0] = cfg.get("initial_conditions", {}).get("a'(0)", 0.0)

    def psi_chi(x):
        return 0.002 * np.cos(0.1 * x) + 0.001 * np.sin(0.2 * x)

    def gamma_zeta(x):
        return packet_value("epsilon", 0.000108071) * np.exp(-alpha * x) + 0.0005 * np.cos(0.03 * x)

    for i in range(1, len(t_vals)):
        ti = t_vals[i - 1]
        sum_term = sum(np.exp(-alpha * j * ti) / delta ** j for j in range(1, n_closure + 1))
        acc = -lam * (a[i - 1] ** 3) + sum_term + psi_chi(ti) + gamma_zeta(ti)
        adot[i] = adot[i - 1] + acc * dt
        a[i] = a[i - 1] + adot[i] * dt

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, a, label="a(t)")
    plt.plot(t_vals, adot, label="a'(t)")
    plt.title("Module A: downstream recursive Friedmann proxy")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Final a({tmax}) = {a[-1]:.6f}")
    print(f"Final a'({tmax}) = {adot[-1]:.6f}")

    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])


def run_module_B(cfg):
    display(Markdown("## Module B: Downstream CP/asymmetry projection"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    eps = cfg.get("epsilon", packet_value("epsilon", 0.000108071))
    lam = cfg.get("lambdaNormalized", packet_value("lambdaNormalized", 0.489442))

    theta = eps * np.sin(lam * t_vals)
    residual = np.gradient(np.gradient(theta, dt), dt) + lam * np.sin(theta)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, theta, label="Theta(t)")
    plt.plot(t_vals, residual, label="residual", alpha=0.7)
    plt.title("Module B: CP phase and residual")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Theta max = {np.max(np.abs(theta)):.9g}")
    print(f"Theta mean abs = {np.mean(np.abs(theta)):.9g}")
    print(f"Residual RMS = {np.sqrt(np.mean(residual ** 2)):.9g}")

    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])


def run_module_C(cfg):
    display(Markdown("## Module C: Ringdown/echo projection"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=100)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    psi = recursive_kernel(t_vals, delta, alpha, nu, n)
    forcing = 0.01 * np.cos(nu * t_vals)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi, label="psi(t)")
    plt.plot(t_vals, forcing, label="F_res(t)", alpha=0.8)
    plt.title("Module C: recursive ringdown echo field")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"psi min = {psi.min():.9g}")
    print(f"psi max = {psi.max():.9g}")
    print(f"psi RMS = {np.sqrt(np.mean(psi ** 2)):.9g}")


def run_module_D(cfg):
    display(Markdown("## Module D: Recursive entropy and identity coherence"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    S = recursive_entropy(t_vals, delta, alpha, cfg.get("nClosure", packet_value("nClosure", 18)))
    psi = recursive_kernel(t_vals, delta, alpha, nu, n)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, S, label="S_rec(t)")
    plt.plot(t_vals, psi, label="psi_self kernel")
    plt.title("Module D: recursive entropy and identity kernel")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Initial S_rec = {S[0]:.9g}")
    print(f"Final S_rec = {S[-1]:.9g}")
    print(f"Final psi = {psi[-1]:.9g}")


def run_module_E(cfg):
    display(Markdown("## Module E: Neural fractal PDE projection"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=100)
    lam = cfg.get("lambda", 0.05)
    phi = np.zeros_like(t_vals)
    phid = np.zeros_like(t_vals)

    for i in range(1, len(t_vals)):
        phi_dd = -lam * phi[i - 1] + 0.01
        phid[i] = phid[i - 1] + phi_dd * dt
        phi[i] = phi[i - 1] + phid[i] * dt

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, phi)
    plt.title("Module E: simplified neural fractal forcing projection")
    plt.xlabel("t")
    plt.ylabel("phi(t)")
    plt.tight_layout()
    plt.show()

    print(f"Final phi = {phi[-1]:.9g}")


def run_module_F(cfg):
    display(Markdown("## Module F: Observer identity stability"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=100)
    beta = cfg.get("beta", 0.8)
    psi = np.zeros_like(t_vals)
    psid = np.zeros_like(t_vals)
    psi[0] = cfg.get("initial_conditions", {}).get("psi(0)", cfg.get("initial_conditions", {}).get("ψ(0)", 0.7))

    def p_chi(x):
        return 0.1 * np.cos(0.2 * x)

    def o_nu(x):
        return 0.01 * np.exp(-0.05 * x)

    for i in range(1, len(t_vals)):
        pdd = -4 * psi[i - 1] ** 3 + beta * psi[i - 1] - p_chi(t_vals[i - 1]) - o_nu(t_vals[i - 1])
        psid[i] = psid[i - 1] + pdd * dt
        psi[i] = psi[i - 1] + psid[i] * dt

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi)
    plt.title("Module F: observer identity stability")
    plt.xlabel("t")
    plt.ylabel("psi_self(t)")
    plt.tight_layout()
    plt.show()

    print(f"Final psi_self = {psi[-1]:.9g}")


def run_module_H(cfg):
    display(Markdown("## Module H: Spin-foam / geometry coherence proxy"))
    print_claim_boundary(cfg)

    delta, alpha, _, _ = get_delta_alpha_nu_n(cfg)
    jmin, jmax = cfg.get("j_range", [1, 40])
    js = np.arange(int(jmin), int(jmax) + 1)
    Sigma = 1 + 0.2 * np.sin(np.pi * js)
    Av = (1 / delta) ** js * np.exp(-alpha * js)
    Z = Sigma * Av * (2 * js + 1)

    plt.figure(figsize=(9, 4))
    plt.plot(js, Z, marker="o")
    plt.title("Module H: recursive spin-foam partition weights")
    plt.xlabel("j")
    plt.ylabel("Z_j")
    plt.tight_layout()
    plt.show()

    print(f"Total partition proxy = {Z.sum():.9e}")
    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])


def run_module_I(cfg):
    display(Markdown("## Module I: Symbolic mass field"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, _, n = get_delta_alpha_nu_n(cfg)
    nu_values = cfg.get("nu_values", [packet_value("nu", 0.00420784)])

    plt.figure(figsize=(9, 4))
    rows = []
    for nu in nu_values:
        psi = recursive_kernel(t_vals, delta, alpha, nu, n)
        m = np.sqrt(np.mean(psi ** 2))
        rows.append({"nu": nu, "rms_mass": m})
        plt.plot(t_vals, psi, label=f"nu={nu}")
    plt.title("Module I: symbolic mass attractor fields")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    display_dict("Mass attractors", rows)


def run_module_J(cfg):
    display(Markdown("## Module J: Observer decoherence divergence"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=100)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    eps = cfg.get("epsilon", packet_value("epsilon", 0.000108071))
    nu_values = cfg.get("nu_values", [nu, nu + eps])
    if len(nu_values) < 2:
        nu_values = [nu, nu + eps]
    nu1, nu2 = nu_values[0], nu_values[1]

    psi1 = recursive_kernel(t_vals, delta, alpha, nu1, n)
    psi2 = recursive_kernel(t_vals, delta, alpha, nu2, n)
    lam_div = np.log(1 + np.abs(psi1 - psi2))
    entropy_gradient = -(psi1 - psi2) ** 2 * np.log(np.abs((psi1 - psi2) ** 2) + 1e-10)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, lam_div, label="lambda_div(t)")
    plt.plot(t_vals, entropy_gradient, label="entropy gradient", alpha=0.8)
    plt.title("Module J: observer branch divergence")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Max lambda_div = {lam_div.max():.9g}")
    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])


def run_module_K(cfg):
    display(Markdown("## Module K: Collapse-rebirth field dynamics"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=120)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    psi_m = recursive_kernel(t_vals, delta, alpha, nu, n)
    psi_anti = recursive_kernel(t_vals, delta, alpha, -nu, n)
    psi_rebirth = recursive_kernel(t_vals, delta, alpha, nu, n, mode="rebirth")
    S_annihil = -(psi_m * psi_anti) * np.log(np.abs(psi_m - psi_anti) + 1e-10)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, psi_rebirth, label="psi_rebirth")
    plt.plot(t_vals, S_annihil, label="S_annihil", alpha=0.8)
    plt.title("Module K: collapse-rebirth and annihilation memory")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Initial psi_rebirth = {psi_rebirth[0]:.9g}")
    print(f"Final psi_rebirth = {psi_rebirth[-1]:.9g}")
    if "expectedDownstreamScreen" in cfg:
        display_dict("Expected downstream screen values", cfg["expectedDownstreamScreen"])


def run_module_L(cfg):
    display(Markdown("## Module L: RFC vs hidden-variable entropy comparison"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=10)
    delta, alpha, _, n = get_delta_alpha_nu_n(cfg)
    nu = cfg.get("nu_values", [packet_value("nu", 0.00420784)])[0]
    lamv = cfg.get("lambda_values", [packet_value("lambdaNormalized", 0.489442)])[0]

    psi_rfc = recursive_kernel(t_vals, delta, alpha, nu, n)
    psi_hv = np.cos(lamv * t_vals + lamv)
    S_rfc = -(psi_rfc ** 2) * np.log(np.abs(psi_rfc ** 2) + 1e-10)
    S_hv = -(psi_hv ** 2) * np.log(np.abs(psi_hv ** 2) + 1e-10)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, S_rfc, label="RFC entropy")
    plt.plot(t_vals, S_hv, label="hidden-variable entropy")
    plt.title("Module L: entropy comparison")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Max S_RFC = {S_rfc.max():.9g}")
    print(f"Max S_HV = {S_hv.max():.9g}")


def run_module_M(cfg):
    display(Markdown("## Module M: Recursive mass and energy emergence"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    psi = recursive_kernel(t_vals, delta, alpha, nu, n)
    dpsi = np.gradient(psi, dt)
    E = dpsi ** 2 + psi ** 2
    m = np.sqrt(np.mean(psi ** 2))
    p2 = np.mean(E) - m ** 2

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, E)
    plt.title("Module M: recursive energy curve")
    plt.xlabel("t")
    plt.ylabel("E(t)")
    plt.tight_layout()
    plt.show()

    print(f"RMS mass = {m:.9g}")
    print(f"Mean energy = {np.mean(E):.9g}")
    print(f"Momentum-squared proxy = {p2:.9g}")


def run_module_O(cfg):
    display(Markdown("## Module O: Baryogenesis from recursive collapse curvature"))
    print_claim_boundary(cfg)

    t_vals, dt, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    psi = recursive_kernel(t_vals, delta, alpha, nu, n)
    kappa = np.gradient(np.gradient(psi, dt), dt)
    a_rec = np.array([sum(np.exp(-alpha * j * t) / delta ** j for j in range(1, n + 1)) for t in t_vals])
    T_rec = np.gradient(np.log(np.abs(a_rec) + 1e-12), dt)
    eta = np.gradient(kappa, dt) / (T_rec ** 3 + 1e-10)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, eta)
    plt.title("Module O: baryon asymmetry proxy")
    plt.xlabel("t")
    plt.ylabel("eta_B/S proxy")
    plt.tight_layout()
    plt.show()

    print(f"Mean eta proxy = {np.mean(eta):.9g}")
    print(f"Max eta proxy = {np.max(eta):.9g}")


def run_module_P(cfg):
    display(Markdown("## Module P: Baryogenesis and dark matter entropy divergence"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=60)
    delta, alpha, _, n = get_delta_alpha_nu_n(cfg)
    nu_values = cfg.get("nu_values", [packet_value("nu", 0.00420784)])

    plt.figure(figsize=(9, 4))
    rows = []
    for nu in nu_values:
        psi_b = recursive_kernel(t_vals, delta, alpha, nu, n)
        psi_d = recursive_kernel(t_vals, delta, alpha, -nu, n)
        missing = psi_b - psi_d
        spike = -(missing ** 2) * np.log(np.abs(missing ** 2) + 1e-10)
        rows.append({"nu": nu, "spike_max": float(np.max(spike)), "missing_mass_rms": float(np.sqrt(np.mean(missing ** 2)))})
        plt.plot(t_vals, spike, label=f"nu={nu}")
    plt.title("Module P: entropy spike")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    display_dict("Entropy divergence metrics", rows)


def run_module_Q(cfg):
    display(Markdown("## Module Q: Scalar-induced decoherence"))
    print_claim_boundary(cfg)

    t_vals, _, _, _ = time_grid(cfg, default_end=100)
    delta, alpha, nu, n = get_delta_alpha_nu_n(cfg)
    m_scalar = cfg.get("parameters", {}).get("m_scalar", 1e-22)

    psi_scalar = np.zeros_like(t_vals)
    for j in range(1, n + 1):
        psi_scalar += (delta ** (-j)) * np.cos(j * t_vals + nu) * np.exp(-alpha * j * t_vals) * np.exp(-m_scalar * t_vals)

    psi_observer = recursive_kernel(t_vals, delta, alpha, nu ** 2, n, mode="sin")
    lam_div = np.log(1 + np.abs(psi_scalar - psi_observer))
    S_psi = -(psi_scalar ** 2) * np.log(np.abs(psi_scalar ** 2) + 1e-10)

    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, lam_div, label="lambda_div")
    plt.plot(t_vals, S_psi, label="S_psi", alpha=0.8)
    plt.title("Module Q: scalar-induced decoherence proxy")
    plt.xlabel("t")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Max lambda_div = {np.max(lam_div):.9g}")

## 6. Router and interactive launcher

In [ ]:
def run_simulation(selected_module):
    cfg = get_config(selected_module)
    mod = cfg.get("module")

    if mod == "G":
        run_module_G(cfg)
    elif mod == "G_legacy":
        run_module_G_legacy(cfg)
    elif mod == "R":
        run_module_R(cfg)
    elif mod == "R_legacy":
        run_module_R_legacy(cfg)
    elif mod == "N":
        run_module_N(cfg)
    elif mod == "N_legacy":
        run_module_N_legacy(cfg)
    elif mod == "S":
        run_module_S(cfg)
    elif mod == "T":
        run_module_T(cfg)
    elif mod == "A":
        run_module_A(cfg)
    elif mod == "B":
        run_module_B(cfg)
    elif mod == "C":
        run_module_C(cfg)
    elif mod == "D":
        run_module_D(cfg)
    elif mod == "E":
        run_module_E(cfg)
    elif mod == "F":
        run_module_F(cfg)
    elif mod == "H":
        run_module_H(cfg)
    elif mod == "I":
        run_module_I(cfg)
    elif mod == "J":
        run_module_J(cfg)
    elif mod == "K":
        run_module_K(cfg)
    elif mod == "L":
        run_module_L(cfg)
    elif mod == "M":
        run_module_M(cfg)
    elif mod == "O":
        run_module_O(cfg)
    elif mod == "P":
        run_module_P(cfg)
    elif mod == "Q":
        run_module_Q(cfg)
    else:
        if str(mod).endswith("_legacy") or cfg.get("currentStatus", "").startswith("deprecated"):
            run_generic_legacy_warning(cfg)
        else:
            display(Markdown(f"## Module {mod}: {cfg.get('description', '')}"))
            print_claim_boundary(cfg)
            display_dict("Configuration", cfg)


def canonical_sort_key(name):
    order = {
        "G": 0,
        "R": 1,
        "N": 2,
        "S": 3,
        "T": 4,
        "A": 10,
        "B": 11,
        "C": 12,
        "D": 13,
        "E": 14,
        "F": 15,
        "H": 16,
        "I": 17,
        "J": 18,
        "K": 19,
        "L": 20,
        "M": 21,
        "O": 22,
        "P": 23,
        "Q": 24,
        "G_legacy": 90,
        "N_legacy": 91,
        "R_legacy": 92
    }
    return (order.get(name, 50), name)


module_options = sorted([module_key(cfg) for cfg in all_configs], key=canonical_sort_key)
default_module = "G" if "G" in module_options else module_options[0]

module_selector = Dropdown(
    options=module_options,
    value=default_module,
    description="Module:",
    layout=Layout(width="440px")
)

run_button = Button(description="Run selected module", button_style="primary")
out = Output()


def on_run_clicked(_):
    out.clear_output(wait=True)
    with out:
        run_simulation(module_selector.value)


run_button.on_click(on_run_clicked)
display(VBox([module_selector, run_button, out]))

display(Markdown(
    "Start with **G**, then **R**, **N**, **S**, and **T**. "
    "Then inspect A-F and H-Q as downstream projections. "
    "Do not use `G_legacy`, `N_legacy`, or `R_legacy` as current derivation modules."
))

## 7. Recommended verification sequence

Use this order when checking the current RFC rebuild:

1. `G` - verify deterministic triadic closure and frozen packet.
2. `R` - verify triad-grouped global closure audit.
3. `N` - verify Module N V2 dimensional projection identities.
4. `S` - verify one-anchor SI bridge.
5. `T` - verify dimensionless coupling map.
6. Downstream modules A-F and H-Q - inspect as projections only.

Do not use `G_legacy`, `N_legacy`, or `R_legacy` as current derivation modules.